# Notebook 01: Dataflow graph and RTL_E002

This notebook uses the shared Phase 3 dataflow path on a real fixture. It loads `tests/fixtures/buggy/buggy_combo_loop.v`, builds the dataflow DAG with `build_dataflow_graph`, and then runs the analysis engine to surface a real `RTL_E002` finding.

In [5]:
from pathlib import Path
import json
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "rtl_analyzer").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook working directory")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rtl_analyzer.dataflow import build_dataflow_graph, find_cycles
from rtl_analyzer.engine import AnalysisEngine
from rtl_analyzer.models import CheckID
from rtl_analyzer.parser import parse_file

fixture_relpath = Path("tests") / "fixtures" / "buggy" / "buggy_combo_loop.v"
fixture_path = repo_root / fixture_relpath
print(fixture_relpath.as_posix())

tests/fixtures/buggy/buggy_combo_loop.v


## Load the real buggy RTL fixture

The fixture is intentionally tiny so the dependency edges are easy to inspect. Signals `a` and `b` feed each other, which should create a combinational loop.

In [6]:
print(fixture_path.read_text())

module buggy_combo_loop(input wire x, output wire y);
    wire a;
    wire b;
    assign a = b;
    assign b = a;
    assign y = a ^ x;
endmodule



## Build the shared dataflow graph

This is the same reusable code path used by the combinational-loop check: `rtl_analyzer.dataflow.build_dataflow_graph` plus `find_cycles`.

In [7]:
parsed = parse_file(fixture_path)
graph = build_dataflow_graph(parsed)

graph_summary = {
    "dependencies": {signal: sorted(deps) for signal, deps in sorted(graph.dependencies.items())},
    "assignment_lines": dict(sorted(graph.assignment_lines.items())),
    "cycles": find_cycles(graph),
}

print(json.dumps(graph_summary, indent=2))

{
  "dependencies": {
    "a": [
      "b"
    ],
    "b": [
      "a"
    ],
    "y": [
      "a",
      "x"
    ]
  },
  "assignment_lines": {
    "a": 4,
    "b": 5,
    "y": 6
  },
  "cycles": [
    [
      "a",
      "b"
    ]
  ]
}


## Run the engine and inspect the real `RTL_E002` finding

The notebook does not reimplement the rule. It calls `AnalysisEngine().analyze_file(...)` so the result stays tied to the same production path used by tests and CLI flows.

In [8]:
result = AnalysisEngine().analyze_file(fixture_path)
rtl_e002_findings = []
for finding in result.findings:
    if finding.check_id != CheckID.RTL_E002:
        continue
    finding_dict = finding.to_dict()
    finding_dict["file"] = fixture_relpath.as_posix()
    rtl_e002_findings.append(finding_dict)

assert rtl_e002_findings, "Expected a real RTL_E002 finding for buggy_combo_loop.v"
print(json.dumps(rtl_e002_findings, indent=2))

[
  {
    "check_id": "RTL_E002",
    "severity": "ERROR",
    "message": "Combinational loop detected across signals ['a', 'b']. The feedback path has no sequential break and can oscillate or settle to an indeterminate value.",
    "file": "tests/fixtures/buggy/buggy_combo_loop.v",
    "line": 4,
    "column": null,
    "fix_hint": "Break the feedback path with registered logic or restructure the combinational assignments to remove the cycle.",
    "source": "rtl_analyzer",
    "confidence": null,
    "metadata": {
      "cycle": [
        "a",
        "b"
      ]
    }
  }
]


## Rerun paths

Re-execute this notebook from the repository root:

```bash
python -m jupyter nbconvert --to notebook --execute --inplace notebooks/rtl_analyzer_phase3/01_dataflow_and_rtl_e002.ipynb
```

Re-run the focused tests that cover the algorithm and notebook contract:

```bash
python -m pytest tests/test_phase3_dataflow.py::test_engine_reports_rtl_e002_on_real_cycle -v
python -m pytest tests/test_phase3_docs.py::test_notebook_01_exists_and_mentions_rtl_e002 -v
```

The implementation path demonstrated here lives in `rtl_analyzer/dataflow.py`, `rtl_analyzer/checks/combinational_loop.py`, and the engine entrypoint in `rtl_analyzer/engine.py`.